In [ ]:
# =========================================================
# 1. Import Libraries
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import OAS

from xgboost import XGBClassifier

from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from rocauc_comparison import delong_roc_test

# CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, LeakyReLU, ReLU, MaxPooling1D
from tensorflow.keras.layers import AveragePooling1D, Activation, Flatten, Dropout, Dense, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import joblib


# =========================================================
# 2. Load Dataset
# =========================================================

data_path = "your/path/SERS_preprocessed_dataset.csv"

df = pd.read_csv(data_path)

X = df.drop(["Groups","Filename"],axis=1)
y = df["Groups"].map({"IGN":0,"IGP":1})


# =========================================================
# 3. Train-Validation Split
# =========================================================

X_train, X_val, y_train, y_val = train_test_split(
X,
y,
test_size=0.20,
stratify=y,
random_state=42
)

print("Train:",X_train.shape)
print("Validation:",X_val.shape)

print("NaN:",np.isnan(X_train).sum().sum())
print("Inf:",np.isinf(X_train).sum().sum())


# =========================================================
# 4. Evaluation Metrics & Bootstrap CI
# =========================================================

def calculate_metrics(y_true,y_pred,y_prob):

    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()

    accuracy=(tp+tn)/(tp+tn+fp+fn)
    sensitivity=tp/(tp+fn)
    specificity=tn/(tn+fp)
    ppv=tp/(tp+fp)
    npv=tn/(tn+fn)
    auc=roc_auc_score(y_true,y_prob)

    return [accuracy,sensitivity,specificity,ppv,npv,auc]


def bootstrap_ci(y_true,y_prob,threshold,n_bootstrap=1000):

    metrics_list=[]
    n=len(y_true)

    for i in range(n_bootstrap):

        idx=np.random.choice(range(n),n,replace=True)

        y_true_b=y_true.iloc[idx]
        y_prob_b=y_prob[idx]

        y_pred_b=(y_prob_b>=threshold).astype(int)

        metrics_list.append(
            calculate_metrics(y_true_b,y_pred_b,y_prob_b)
        )

    metrics_array=np.array(metrics_list)

    lower=np.percentile(metrics_array,2.5,axis=0)
    upper=np.percentile(metrics_array,97.5,axis=0)

    return lower,upper


metrics_names=["Accuracy","Sensitivity","Specificity","PPV","NPV","AUC"]


# =========================================================
# 5. Model Definitions
# =========================================================

models={

"LR":LogisticRegression(max_iter=1000),

"LDA":LinearDiscriminantAnalysis(),

"RF":RandomForestClassifier(random_state=42, n_jobs=-1),

"SVM":SVC(probability=True,random_state=42),

"KNN":KNeighborsClassifier(),

"XGB":XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

}

oas_estimator = OAS(store_precision=False, assume_centered=False)


# =========================================================
# 6. Hyperparameter Grid
# =========================================================

param_grid = {
# ==========================================
# Logistic Regression
# ==========================================
"LR":[
    {
        "model__penalty":["l1","l2"],
        "model__C":[0.01,0.1,1,10],
        "model__solver":["liblinear"],
        "model__class_weight":[None,"balanced"]
    },
    {
        "model__penalty":["l1","l2"],
        "model__C":[0.01,0.1,1,10],
        "model__solver":["saga"],
        "model__class_weight":[None,"balanced"]
    },
    {
        "model__penalty":["l2"],
        "model__C":[0.01,0.1,1,10],
        "model__solver":["lbfgs"],
        "model__class_weight":[None,"balanced"]
    }
],


# ==========================================
# LDA
# ==========================================
"LDA":[
    {"model__solver":["svd"]},

    {
        "model__solver":["lsqr"],
        "model__shrinkage":["auto",0.1,0.5,0.9]
    },

    {
        "model__solver":["lsqr"],
        "model__covariance_estimator":[oas_estimator]
    }
],


# ==========================================
# Random Forest
# ==========================================
"RF":{
    "model__n_estimators":[200,400],
    "model__max_depth":[10,20,30,None],
    "model__min_samples_split":[2,5,10],
    "model__min_samples_leaf":[1,2,4]
},


# ==========================================
# SVM
# ==========================================
"SVM":[
    {
        "model__kernel":["linear"],
        "model__C":[0.1,1,10,100],
        "model__class_weight":[None,"balanced"]
    },
    {
        "model__kernel":["rbf"],
        "model__C":[0.1,1,10,100],
        "model__gamma":[1,0.1,0.01,0.001],
        "model__class_weight":[None,"balanced"]
    },
    {
        "model__kernel":["poly"],
        "model__C":[0.1,1,10,100],
        "model__gamma":[1,0.1,0.01],
        "model__degree":[2,3,4],
        "model__class_weight":[None,"balanced"]
    }
],


# ==========================================
# KNN
# ==========================================
"KNN":[
    {
        "model__n_neighbors":[3,5,7],
        "model__weights":["uniform","distance"],
        "model__metric":["euclidean","manhattan"]
    },
    {
        "model__n_neighbors":[3,5,7],
        "model__weights":["uniform","distance"],
        "model__metric":["minkowski"],
        "model__p":[1,2]
    }
],


# ==========================================
# XGBoost
# ==========================================
"XGB":{
    "model__n_estimators":[200,400],
    "model__max_depth":[3,4,6],
    "model__learning_rate":[0.03,0.05,0.1]
}


}


# =========================================================
# 7. Cross-validation Strategy
# =========================================================

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

training_metrics={}
training_ci={}
roc_data={}
best_models={}
cv_conf_matrices={}
cv_probabilities = {}
cv_probabilities_ign = {}
threshold_fixed = 0.5


# =========================================================
# 8. Training ML Models (GridSearch + CV)
# =========================================================

for name,model in models.items():

    print("\nTraining:",name)

    if name in ["XGB","RF"]:
        pipe=Pipeline([("model",model)])
    else:
        pipe=Pipeline([
        ("scaler",StandardScaler()),
        ("model",model)
        ])

    grid=GridSearchCV(
    pipe,
    param_grid[name],
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    error_score="raise"
    )

    grid.fit(X_train,y_train)

    best_model=grid.best_estimator_
    best_models[name]=best_model

    best_index = grid.best_index_

    mean_auc = grid.cv_results_["mean_test_score"][best_index] 
    std_auc = grid.cv_results_["std_test_score"][best_index] 
    
    print(f"Best CV AUC: {mean_auc:.3f} ± {std_auc:.3f}")
    print("Best parameters:", grid.best_params_)

    results_df = pd.DataFrame(grid.cv_results_)
    results_df.to_csv(f"D:/TB-SERS Analysis/gridsearch_{name}.csv",index=False)

    y_prob_all = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
    )

    y_prob_cv = y_prob_all[:,1]
    y_prob_cv_ign = y_prob_all[:,0]
    
    cv_probabilities[name] = y_prob_cv
    cv_probabilities_ign[name] = y_prob_cv_ign

    fpr,tpr,thr=roc_curve(y_train,y_prob_cv)

    y_pred_cv=(y_prob_cv >= threshold_fixed).astype(int)

    cm_cv = confusion_matrix(y_train, y_pred_cv)
    cv_conf_matrices[name] = cm_cv

    metrics=calculate_metrics(y_train,y_pred_cv,y_prob_cv)
    training_metrics[name]=metrics

    lower,upper=bootstrap_ci(
    y_train.reset_index(drop=True),
    y_prob_cv,
    threshold_fixed
    )

    training_ci[name]=(lower,upper)
    roc_data[name]=(fpr,tpr)


# =========================================================
# 9. CNN Model Definition
# =========================================================

def create_cnn_model(input_shape):

    model = Sequential([
        Conv1D(16,3,padding="same",input_shape=input_shape),
        BatchNormalization(),
        LeakyReLU(),
        AveragePooling1D(2),

        Conv1D(32,3,padding="same"),
        BatchNormalization(),
        LeakyReLU(),
        AveragePooling1D(2),

        Conv1D(64,3,padding="same"),
        BatchNormalization(),
        LeakyReLU(),
        AveragePooling1D(2),

        Conv1D(128,3,padding="same"),
        BatchNormalization(),
        LeakyReLU(),
        AveragePooling1D(2),

        Flatten(),
        Dense(128,activation='relu'),
        Dense(1,activation='sigmoid')
    ])

    optimizer = Adam(learning_rate=1e-4)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
#PART 2: CNN CV + Training Summary + ROC + Model Selection

# =========================================================
# 10. CNN Cross-validation
# =========================================================

X_train_cnn=X_train.values.reshape(X_train.shape[0],X_train.shape[1],1)

y_prob_cv=np.zeros(len(y_train))

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_cnn, y_train)):

    print("CNN Fold:", fold + 1)

    X_tr = X_train_cnn[train_idx]
    X_valcnn = X_train_cnn[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_valcnn = y_train.iloc[val_idx]

    model = create_cnn_model((X_train_cnn.shape[1], 1))

    model.fit(
        X_tr,
        y_tr,
        validation_data=(X_valcnn, y_valcnn),
        epochs=100,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )

    y_prob = model.predict(X_valcnn).ravel()

    auc_fold = roc_auc_score(y_valcnn, y_prob)
    aucs.append(auc_fold)

    y_prob_cv[val_idx] = y_prob

mean_auc = np.mean(aucs)
std_auc = np.std(aucs)

print(f"CNN AUC = {mean_auc:.3f} ± {std_auc:.3f}")

y_prob_cv_ign = 1 - y_prob_cv

fpr,tpr,thr=roc_curve(y_train,y_prob_cv)

y_pred_cv=(y_prob_cv>=threshold_fixed).astype(int)

cm_cv = confusion_matrix(y_train, y_pred_cv)
cv_conf_matrices["CNN"] = cm_cv

metrics=calculate_metrics(y_train,y_pred_cv,y_prob_cv)
training_metrics["CNN"]=metrics

lower,upper=bootstrap_ci(
y_train.reset_index(drop=True),
y_prob_cv,
threshold_fixed
)

training_ci["CNN"]=(lower,upper)

roc_data["CNN"]=(fpr,tpr)

cv_probabilities["CNN"] = y_prob_cv
cv_probabilities_ign["CNN"] = y_prob_cv_ign


# =========================================================
# 11. Training Performance Summary (CV + 95% CI)
# =========================================================

ci_rows=[]

for model in training_ci:

    lower,upper=training_ci[model]

    row={}

    for i,m in enumerate(metrics_names):

        if m=="AUC":
            row[m]=f"{training_metrics[model][i]:.3f} ({lower[i]:.2f}-{upper[i]:.2f})"
        else:
            row[m]=f"{training_metrics[model][i]*100:.2f}% ({lower[i]:.2f}-{upper[i]:.2f})"

    ci_rows.append(row)

training_table=pd.DataFrame(
ci_rows,
index=training_metrics.keys()
)

print("\nTraining Performance (CV + 95% CI)")
print(training_table)


# =========================================================
# 12. ROC Curve (Training)
# =========================================================

plt.figure(figsize=(7,6))

desired_order = ['CNN', 'LR', 'LDA', 'SVM', 'KNN', 'RF', 'XGB']

for name in desired_order:
    if name in roc_data:
        fpr, tpr = roc_data[name]
        auc = training_metrics[name][-1]
        
        plt.plot(
            fpr, 
            tpr, 
            label=f"{name} (AUC={auc:.3f})"
        )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Training - 5-fold CV)")

plt.legend()

plt.tight_layout()
plt.show()


# =========================================================
# 13. Model Selection (Best Model + DeLong Test)
# =========================================================

best_model_name=max(training_metrics,key=lambda x:training_metrics[x][-1])

print("\nBest Model:",best_model_name)

best = best_model_name

for name in cv_probabilities:
    if name == best:
        continue

    p_log = delong_roc_test(
        y_train,
        cv_probabilities[best],
        cv_probabilities[name]
    )

    auc_sbest = training_metrics[best][-1]
    auc_other = training_metrics[name][-1]

    if isinstance(p_log, np.ndarray):
        p_log = p_log.item()

    p = 10 ** p_log

    print(f"{best} vs {name} (AUC={auc_other:.3f}) -> p = {p:.4e}")

In [ ]:
#PART 3: Threshold + Confidence Tier + Calibration

# =========================================================
# 14. Threshold Optimization (Youden Index)
# =========================================================

y_prob_cv_best = cv_probabilities[best_model_name]

fpr, tpr, thr = roc_curve(y_train, y_prob_cv_best)

youden = tpr - fpr
best_idx_youden = np.argmax(youden)
threshold = thr[best_idx_youden]


# =========================================================
# 15. Confidence Analysis (Margin + Tier)
# =========================================================

margin = np.abs(y_prob_cv_best - threshold)

df_plot = pd.DataFrame({
    "margin": margin,
    "y_true": y_train,
})

low_cut = np.percentile(margin, 30)
high_cut = np.percentile(margin, 70)

print(f"Low   : < {low_cut:.4f}")
print(f"Medium: {low_cut:.4f} – {high_cut:.4f}")
print(f"High  : > {high_cut:.4f}")

df_plot["tier"] = pd.cut(
    df_plot["margin"],
    bins=[-np.inf, low_cut, high_cut, np.inf],
    labels=["Low", "Medium", "High"]
)

df_plot["y_pred"] = (y_prob_cv_best >= threshold).astype(int)

for tier in ["Low", "Medium", "High"]:
    subset = df_plot[df_plot["tier"] == tier]
    acc = accuracy_score(subset["y_true"], subset["y_pred"])
    print(tier, acc, "n=", len(subset))

df_plot["correct"] = (df_plot["y_true"] == df_plot["y_pred"]).astype(int)

print(df_plot.groupby("tier")["correct"].mean())


# =========================================================
# 16. Margin Distribution Plot
# =========================================================

sns.set_theme(style="ticks")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 1.2,
})

plt.figure(figsize=(5,4))

sns.kdeplot(
    df_plot["margin"],
    fill=True,
    color="#4d4d4d",
    linewidth=1.5
)

plt.axvline(low_cut, linestyle="--", linewidth=1.2, color="red")
plt.axvline(high_cut, linestyle="--", linewidth=1.2, color="red")

plt.xlabel("Margin (|Predicted probability − threshold|)", labelpad=8)
plt.ylabel("Density", labelpad=8)
plt.title("Distribution of Confidence Margin", pad=10)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(axis='both', direction='out', length=4, width=1)

plt.tight_layout()
plt.show()


# =========================================================
# 17. Accuracy by Confidence Tier (Training)
# =========================================================

order = ["Low", "Medium", "High"]

plt.figure(figsize=(5,4))

ax = sns.barplot(
    data=df_plot,
    x="tier",
    y="correct",
    order=order,
    ci=95,
    capsize=0.1,
    errwidth=1.2
)

palette = ["#D3D3D3", "#A9A9A9", "#404040"]
for i, bar in enumerate(ax.patches):
    bar.set_facecolor(palette[i])
    bar.set_edgecolor("black")
    bar.set_linewidth(1)

ax.set_ylabel("Accuracy", labelpad=10)
ax.set_xlabel("")
ax.set_title("Accuracy by Confidence Tier", pad=12)

ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(axis='both', direction='out', length=4, width=1)

plt.tight_layout()
plt.show()


# =========================================================
# 18. ROC Curve (Training CV - Best Model + Threshold)
# =========================================================

thr_idx = np.argmin(np.abs(thr - threshold))

sens = tpr[thr_idx]
spec = 1 - fpr[thr_idx]

plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    lw=2,
    color="#8c564b",
    label=f"{best_model_name} (AUC={training_metrics[best_model_name][-1]:.3f})"
)

plt.scatter(
    fpr[thr_idx],
    tpr[thr_idx],
    color="red",
    s=80,
    zorder=3
)

plt.annotate(
    f"Threshold = {threshold:.3f}\nSensitivity = {sens:.3f}\nSpecificity = {spec:.3f}",
    xy=(0.98,0.05),
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=10,
    bbox=dict(boxstyle="round", fc="white", ec="gray")
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(f"ROC Curve (Training CV - {best_model_name})")

plt.legend()

plt.tight_layout()
plt.show()


# =========================================================
# 19. Probability Calibration (Training)
# =========================================================

from sklearn.isotonic import IsotonicRegression

y_prob_cal = np.zeros_like(y_prob_cv_best)

for train_idx, val_idx in cv.split(y_prob_cv_best, y_train):

    prob_tr = y_prob_cv_best[train_idx]
    prob_val = y_prob_cv_best[val_idx]

    y_tr = y_train.iloc[train_idx]

    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(prob_tr, y_tr)

    y_prob_cal[val_idx] = ir.transform(prob_val)


# ===== BEFORE =====
prob_true, prob_pred = calibration_curve(
    y_train, y_prob_cv_best, n_bins=10
)

# ===== AFTER =====
prob_true_cal, prob_pred_cal = calibration_curve(
    y_train, y_prob_cal, n_bins=10
)

plt.figure(figsize=(5,4))
plt.plot([0,1],[0,1],'--',color='gray')

plt.plot(prob_pred, prob_true, marker='o', label='Before', color='black')
plt.plot(prob_pred_cal, prob_true_cal, marker='o', label='After', color='#d62728')

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("calibration_curve CV Train")

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
#PART 4: Final Model + Validation + Blind Test

# =========================================================
# 20. Confusion Matrix (Training CV - All Models)
# =========================================================

for model_name, cm in cv_conf_matrices.items():

    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["IGN", "IGP"]
    )

    disp.plot(
        cmap="Blues",
        values_format="d",
        colorbar=False
    )

    plt.title(
        f"{model_name}\nSensitivity {sensitivity*100:.2f}% | Specificity {specificity*100:.2f}%"
    )

    plt.tight_layout()
    plt.show()


# =========================================================
# 21. Final Model Training
# =========================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=100,
    restore_best_weights=True
)

if best_model_name=="CNN":

    final_model=create_cnn_model((X_train_cnn.shape[1],1))

    history = final_model.fit(
        X_train_cnn,
        y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )

    history_df = pd.DataFrame(history.history)

    model_path = "your/path/final_cnn_model.h5"
    final_model.save(model_path)

    print(f"Model saved to: {model_path}")

    loss = history.history['loss']
    val_loss = history.history['val_loss']

    acc = history.history.get('accuracy')
    val_acc = history.history.get('val_accuracy')

    epochs = range(1, len(loss) + 1)

    best_epoch = np.argmin(val_loss) + 1
    print("Best epoch:", best_epoch)

    plt.figure(figsize=(8,6))

    # Loss
    plt.subplot(2,1,1)
    plt.plot(epochs, loss, label='Train', linewidth=2, color='#ff7f0e')
    plt.plot(epochs, val_loss, label='Validation', linewidth=2, color='#1f77b4')
    plt.axvline(best_epoch, linestyle='--', alpha=0.7)

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss')
    plt.legend(frameon=False)

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    # Accuracy
    plt.subplot(2,1,2)
    if acc is not None:
        plt.plot(epochs, acc, label='Train', linewidth=2, color='#ff7f0e')
        plt.plot(epochs, val_acc, label='Validation', linewidth=2, color='#1f77b4')
        plt.axvline(best_epoch, linestyle='--', alpha=0.7)

        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Accuracy')
        plt.legend(frameon=False)

        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)

    plt.tight_layout(pad=0.8)
    plt.show()

    X_val_cnn=X_val.values.reshape(X_val.shape[0],X_val.shape[1],1)

    y_prob=final_model.predict(X_val_cnn).ravel()

else:

    final_model=best_models[best_model_name]

    final_model.fit(X_train,y_train)

    y_prob=final_model.predict_proba(X_val)[:,1]


y_pred=(y_prob>=threshold).astype(int)
y_pred_val = (y_prob >= threshold).astype(int)


# =========================================================
# 22. Validation Prediction Table
# =========================================================

val_result = pd.DataFrame({
    "True": y_val,
    "Pred": y_pred_val,
    "Prob_IGP": y_prob,
    "Prob_IGN": 1 - y_prob
})

val_result["Correct"] = val_result["True"] == val_result["Pred"]

val_result["Prob_predicted"] = np.where(
    val_result["Pred"] == 1,
    val_result["Prob_IGP"],
    val_result["Prob_IGN"]
)

val_result["Margin"] = np.abs(
    val_result["Prob_predicted"] - threshold
)

val_result["Tier"] = pd.cut(
    val_result["Margin"],
    bins=[-np.inf, low_cut, high_cut, np.inf],
    labels=["Low", "Medium", "High"]
)


# =========================================================
# 23. Validation Calibration
# =========================================================

ir = IsotonicRegression(out_of_bounds='clip')
ir.fit(y_prob_cv_best, y_train)

y_prob_val_cal = ir.transform(y_prob)

prob_true_val, prob_pred_val = calibration_curve(
    y_val, y_prob, n_bins=10,
    strategy='quantile'
)

prob_true_val_cal, prob_pred_val_cal = calibration_curve(
    y_val, y_prob_val_cal, n_bins=10,
    strategy='quantile'
)

plt.figure(figsize=(5,4))

plt.plot([0,1],[0,1],'--',color='gray', label="Perfectly calibrated")

plt.plot(prob_pred_val, prob_true_val, marker='o', label='Before', color='black')
plt.plot(prob_pred_val_cal, prob_true_val_cal, marker='o', label='After', color='#d62728')

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve (Validation)")

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


# =========================================================
# 24. Validation Tier Analysis
# =========================================================

acc_by_tier = val_result.groupby("Tier")["Correct"].mean()
count_by_tier = val_result["Tier"].value_counts()

result = pd.DataFrame({
    "Accuracy": acc_by_tier,
    "n": count_by_tier
})

print(result)

order = ["Low", "Medium", "High"]

plt.figure(figsize=(5,4))

ax = sns.barplot(
    data=val_result,
    x="Tier",
    y="Correct",
    order=order,
    ci=95,
    capsize=0.1,
    errwidth=1.2
)

palette = ["#D3D3D3", "#A9A9A9", "#404040"]

for i, bar in enumerate(ax.patches):
    bar.set_facecolor(palette[i])
    bar.set_edgecolor("black")
    bar.set_linewidth(1)

ax.set_ylabel("Accuracy", labelpad=10)
ax.set_xlabel("")
ax.set_title("Validation: Accuracy by Confidence Tier", pad=12)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

ax.tick_params(axis='both', direction='out', length=4, width=1)

plt.tight_layout()
plt.show()

print("\nValidation Prediction Detail")
print(val_result)


# =========================================================
# 25. Validation Performance
# =========================================================

val_metrics=calculate_metrics(y_val,y_pred,y_prob)

lower,upper=bootstrap_ci(
y_val.reset_index(drop=True),
y_prob,
threshold
)

val_row={}

for i,m in enumerate(metrics_names):

    if m=="AUC":
        val_row[m]=f"{val_metrics[i]:.3f} ({lower[i]:.2f}-{upper[i]:.2f})"
    else:
        val_row[m]=f"{val_metrics[i]*100:.2f}% ({lower[i]:.2f}-{upper[i]:.2f})"

val_table=pd.DataFrame(
[val_row],
columns=metrics_names,
index=["Validation"]
)

print("\nValidation Performance (95% CI)")
print(val_table)


# =========================================================
# 26. ROC Curve (Validation)
# =========================================================

fpr_val, tpr_val, _ = roc_curve(y_val, y_prob)

plt.figure(figsize=(6,5))

plt.plot(
    fpr_val,
    tpr_val,
    color="#8c564b",
    lw=2,
    label=f"Validation (AUC = {val_metrics[-1]:.3f})"
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (Validation - {best_model_name})")

plt.legend()

plt.tight_layout()
plt.show()


# =========================================================
# 27. Confusion Matrix (Validation)
# =========================================================

cm_val = confusion_matrix(y_val, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=["IGN","IGP"]
)

disp.plot(cmap="Greens", values_format="d")

plt.title("Confusion Matrix (Validation)")
plt.tight_layout()
plt.show()


# =========================================================
# 28. Blind Test Evaluation
# =========================================================

blind_path = "your/path/SERS_preprocessed_dataset.csv"

blind= pd.read_csv(blind_path)

X_blind = blind.drop(["Groups","Filename"],axis=1)
y_blind = blind["Groups"].map({"IGN":0,"IGP":1})

if best_model_name=="CNN":

    X_blind_cnn = X_blind.values.reshape(X_blind.shape[0],X_blind.shape[1],1)

    y_prob_blind = final_model.predict(X_blind_cnn).ravel()
    y_prob_blind_ign = 1 - y_prob_blind

else:
    y_prob_all_blind = final_model.predict_proba(X_blind)

    y_prob_blind = y_prob_all_blind[:,1]
    y_prob_blind_ign = y_prob_all_blind[:,0]


y_pred_blind = (y_prob_blind >= threshold).astype(int)

# =========================================================
# 29. Blind Performance
# =========================================================

blind_metrics = calculate_metrics(y_blind, y_pred_blind, y_prob_blind)

lower, upper = bootstrap_ci(
    y_blind.reset_index(drop=True),
    y_prob_blind,
    threshold
)

blind_row = {}

for i,m in enumerate(metrics_names):

    if m=="AUC":
        blind_row[m]=f"{blind_metrics[i]:.3f} ({lower[i]:.2f}-{upper[i]:.2f})"
    else:
        blind_row[m]=f"{blind_metrics[i]*100:.2f}% ({lower[i]:.2f}-{upper[i]:.2f})"

blind_table = pd.DataFrame(
    [blind_row],
    columns=metrics_names,
    index=["Blind"]
)

print("\nBlind Performance (95% CI)")
print(blind_table)


# =========================================================
# 30. ROC Curve (Blind)
# =========================================================

fpr_blind, tpr_blind, _ = roc_curve(y_blind, y_prob_blind)

plt.figure(figsize=(6,5))

plt.plot(
    fpr_blind,
    tpr_blind,
    color="#8c564b",
    lw=2,
    label=f"Blind (AUC = {blind_metrics[-1]:.3f})"
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (Blind - {best_model_name})")

plt.legend()

plt.tight_layout()
plt.show()


# =========================================================
# 31. Confusion Matrix (Blind)
# =========================================================

cm_blind = confusion_matrix(y_blind, y_pred_blind)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_blind,
    display_labels=["IGN","IGP"]
)

disp.plot(cmap="Reds", values_format="d")

plt.title("Confusion Matrix (Blind)")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Import
# ============================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, AveragePooling1D
from tensorflow.keras.layers import BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import MaxPooling1D

# ============================================
# Load Dataset
# ============================================
df_path = "your/path/SERS_preprocessed_dataset.csv"

df = pd.read_csv(df_path)

X = df.drop(["Groups", "Filename"], axis=1)
y = df["Groups"].map({"IGN": 0, "IGP": 1})

# ============================================
# Split
# ============================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ============================================
# Prepare CNN
# ============================================
X_train_np = X_train.values[..., np.newaxis]
y_train_np = y_train.values

input_shape = X_train_np.shape[1:]

# ============================================
# Model Function
# ============================================
def create_cnn_model(input_shape, filters_list, kernel_size, pooling, dense_size):

    model = Sequential()

    for i, f in enumerate(filters_list):

        if i == 0:
            model.add(Conv1D(f, kernel_size, padding="same", input_shape=input_shape))
        else:
            model.add(Conv1D(f, kernel_size, padding="same"))

        model.add(BatchNormalization())
        model.add(LeakyReLU())

        if pooling == "avg":
            model.add(AveragePooling1D(2))
        else:
            model.add(MaxPooling1D(2))

    model.add(Flatten())
    model.add(Dense(dense_size, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='binary_crossentropy'
    )

    return model

# ============================================
# CV Function
# ============================================
def evaluate_model(X, y, input_shape, filters_list, kernel_size, pooling, dense_size, batch_size):

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, val_idx in kf.split(X, y):

        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = create_cnn_model(
            input_shape,
            filters_list,
            kernel_size,
            pooling,
            dense_size
        )

        model.fit(
            X_tr, y_tr,
            epochs=20,
            batch_size=batch_size,
            verbose=0
        )

        y_prob = model.predict(X_val).ravel()
        aucs.append(roc_auc_score(y_val, y_prob))

    return np.mean(aucs), np.std(aucs)

# =========================================================
# Config Sets
# =========================================================

# 🔥 Table S1
configs_full = [
    [16],
    [32],
    [64],
    [128],
    [16,16],
    [32,32],
    [64,64],
    [128,128],
    [16,32],
    [16,16,16],
    [32,32,32],
    [64,64,64],
    [128,128,128],
    [16,32,64],
    [16,16,16,16],
    [32,32,32,32],
    [64,64,64,64],
    [128,128,128,128],
    [16,32,64,128],
]

# 🔥 Table S2–S4
configs_reduced = [
    [16],
    [16,32],
    [16,32,64],
    [16,32,64,128],
]

# =========================================================
# TABLE S1: Architecture (Full Search)
# =========================================================
results_s1 = []

print("\n===== TABLE S1: Architecture (Full) =====")

for filters_list in configs_full:

    print("Filters:", filters_list)

    mean_auc, std_auc = evaluate_model(
        X_train_np, y_train_np, input_shape,
        filters_list,
        kernel_size=3,
        pooling="avg",
        dense_size=128,
        batch_size=16
    )

    print(f" → AUC = {mean_auc:.3f} ± {std_auc:.3f}")

    results_s1.append({
        "filters": str(filters_list),
        "AUC": f"{mean_auc:.3f}±{std_auc:.3f}"
    })

df_s1 = pd.DataFrame(results_s1)

# 🔥 select best from FULL search
best_filters = eval(
    df_s1.loc[
        df_s1["AUC"].str.split("±").str[0].astype(float).idxmax(),
        "filters"
    ]
)

print("\nBest filters (from S1):", best_filters)

# =========================================================
# TABLE S2: Kernel Size (Reduced)
# =========================================================
kernels = [3,5,7,9]
results_s2 = []

print("\n===== TABLE S2: Kernel Size =====")

for k in kernels:

    mean_auc, std_auc = evaluate_model(
        X_train_np, y_train_np, input_shape,
        best_filters,  
        kernel_size=k,
        pooling="avg",
        dense_size=128,
        batch_size=16
    )

    print(f"k={k} → {mean_auc:.3f} ± {std_auc:.3f}")

    results_s2.append({
        "kernel": k,
        "AUC": f"{mean_auc:.3f}±{std_auc:.3f}"
    })

df_s2 = pd.DataFrame(results_s2)

best_kernel = df_s2.loc[
    df_s2["AUC"].str.split("±").str[0].astype(float).idxmax(),
    "kernel"
]

# =========================================================
# TABLE S3: Pooling
# =========================================================
poolings = ["avg", "max"]
results_s3 = []

print("\n===== TABLE S3: Pooling =====")

for p in poolings:

    mean_auc, std_auc = evaluate_model(
        X_train_np, y_train_np, input_shape,
        best_filters,
        kernel_size=best_kernel,
        pooling=p,
        dense_size=128,
        batch_size=16
    )

    print(f"{p} → {mean_auc:.3f} ± {std_auc:.3f}")

    results_s3.append({
        "pooling": p,
        "AUC": f"{mean_auc:.3f}±{std_auc:.3f}"
    })

df_s3 = pd.DataFrame(results_s3)

best_pooling = df_s3.loc[
    df_s3["AUC"].str.split("±").str[0].astype(float).idxmax(),
    "pooling"
]

# =========================================================
# TABLE S4: Dense Layer
# =========================================================
dense_sizes = [32, 64,128,256]
results_s4 = []

print("\n===== TABLE S4: Dense Layer =====")

for d in dense_sizes:

    mean_auc, std_auc = evaluate_model(
        X_train_np, y_train_np, input_shape,
        best_filters,
        kernel_size=best_kernel,
        pooling=best_pooling,
        dense_size=d,
        batch_size=16
    )

    print(f"d={d} → {mean_auc:.3f} ± {std_auc:.3f}")

    results_s4.append({
        "dense": d,
        "AUC": f"{mean_auc:.3f}±{std_auc:.3f}"
    })

df_s4 = pd.DataFrame(results_s4)

best_dense = df_s4.loc[
    df_s4["AUC"].str.split("±").str[0].astype(float).idxmax(),
    "dense"
]